# nemo-endpoints-test

NeMo Microservices project on the Miramar platform.

- Edit `job_config.yaml` to configure your training job
- Use the cells below to interact with NeMo directly
- Use **Deploy to NeMo** (GitHub Actions) for CI/CD submission

**Requires SSH tunnel:** `ssh -L 8082:localhost:8082 spark-79b7.local`  
**Requires laptop `/etc/hosts`:** `127.0.0.1 nemo.test nim.test data-store.test`


In [ ]:
# Install NeMo SDK if needed
# !pip install nemo-microservices pyyaml

from nemo_microservices import NeMoMicroservices

client = NeMoMicroservices(
    base_url='http://nemo.test:8082',
    inference_base_url='http://nim.test:8082',
)
print('Connected to NeMo')

In [ ]:
# List available namespaces and base models
namespaces = client.namespaces.list()
print('Namespaces:', [n.name for n in namespaces.items])

models = client.models.list()
print('\nAvailable base models:')
for m in models.items:
    print(f'  {m.name}')

In [ ]:
# Load job config
import yaml
with open('job_config.yaml') as f:
    config = yaml.safe_load(f)
print(yaml.dump(config))

In [ ]:
# Submit a customization (fine-tuning) job
job = client.customization.jobs.create(
    name=config['name'],
    model=config['model'],
    training_config={
        'num_epochs': config['training']['num_epochs'],
        'batch_size': config['training'].get('batch_size', 8),
        'learning_rate': config['training'].get('learning_rate', 1e-4),
    },
    dataset={'file_path': config['training'].get('dataset_path', '')},
)
print(f'Job submitted: {job.id}  status: {job.status}')

In [ ]:
# Monitor job status
import time
job_name = config['name']  # or paste a job name here

for _ in range(30):
    j = client.customization.jobs.retrieve(job_name)
    print(f'  {j.status}')
    if j.status in ('completed', 'failed', 'cancelled'):
        break
    time.sleep(30)

In [ ]:
# List all jobs
jobs = client.customization.jobs.list()
for j in jobs.items:
    print(f'{j.name:40s} {j.status}')